In [0]:
import dlt
from pyspark.sql.functions import col, lit, concat, trim, upper, when, coalesce

In [0]:
# ============================================================================
# SOURCE VIEW: Combine violation data from both Silver tables
# This is the "incoming" data that DLT will compare against existing records
# to detect changes and apply SCD Type 2 logic
# ============================================================================

@dlt.view("silver_violations_combined")
def silver_violations_combined():
    chicago = (
        spark.table("workspace.default.silver_chicago_inspections")
        .where(col("violation_code").isNotNull() & (col("violation_code") != ""))
        .select("source_city", "violation_code", "violation_description", "violation_detail", "violation_severity")
    )
    
    dallas = (
        spark.table("workspace.default.silver_dallas_inspections")
        .where(col("violation_code").isNotNull() & (col("violation_code") != ""))
        .select("source_city", "violation_code", "violation_description", "violation_detail", "violation_severity")
    )
    
    return chicago.union(dallas).dropDuplicates(["violation_code", "source_city", "violation_description"])

In [0]:
# ============================================================================
# DLT SCD TYPE 2: dim_violation
# 
# Tracks historical changes to violation descriptions over time.
# Natural key: (violation_code, source_city)
# Tracked columns: violation_description, violation_detail, violation_severity
#
# When a violation's description or detail changes, DLT will:
#   1. Set end_date on the old record and mark is_current = false
#   2. Insert a new record with the updated values and is_current = true
# ============================================================================

dlt.create_streaming_table(
    name="dim_violation_scd2",
    comment="SCD Type 2 dimension for violation codes - tracks description changes over time",
    table_properties={"quality": "gold"}
)

dlt.apply_changes(
    target="dim_violation_scd2",
    source="silver_violations_combined",
    keys=["violation_code", "source_city"],
    sequence_by=lit(1),  # No natural sequence field; all records treated as latest
    stored_as_scd_type=2,
    columns=["violation_code", "source_city", "violation_description", "violation_detail", "violation_severity"]
)